In [19]:
import arcpy
import sys
import os


arcpy.env.overwriteOutput = True

def getGDB(input_gdb):
    gdb_list= []
    arcpy.env.workspace= input_gdb
    fc_list=  arcpy.ListFeatureClasses()
    rast_list= arcpy.ListRasters()
    if fc_list:
        for fc in fc_list: gdb_list.append(fc)
    if rast_list:
        for r in rast_list: gdb_list.append(r)
    return gdb_list

def checkGDB(output_folder, output_gdb):
    if not output_gdb.endswith('.gdb'):      
        output_gdb= output_gdb+'.gdb'
        if output_folder:
            output_gdb= output_folder+'\\'+output_gdb
            return output_gdb
        else: return output_gdb
    else: 
        if output_folder:
            output_gdb= output_folder+'\\'+output_gdb
            return output_gdb
        else: return output_gdb

def checkGPKG(output_folder, output_gpkg):
    if not output_gpkg.endswith('.gpkg'):      
        output_gpkg= output_gpkg+'.gpkg'
        if output_folder:
            output_gpkg= output_folder+'\\'+output_gpkg
            return output_gpkg
        else: return output_gpkg
    else: 
        if output_folder:
            output_gpkg= output_folder+'\\'+output_gpkg
            return output_gpkg
        else: return output_gpkg

def setName(environment, nom_gdb, output_folder='',map_name=''):
    #l'environment és l'entorn del que surten les dades originalment. pot ser o (.gdb) o (.aprx)
    #output_gdb demana el nom de la ruta de sortida de la geodatabase resultant
    output_gdb= checkGDB(output_folder, nom_gdb)
    if os.path.isfile(output_gdb) == False:
        new_list= []

        if environment.endswith('.gdb'):
            arcpy.Copy_management(environment, output_gdb)
            gdb_list= getGDB(output_gdb)
            new_list= gdb_list

        elif environment.endswith('.aprx'):           
            aprx= arcpy.mp.ArcGISProject(environment)
            #aprx= arcpy.mp.ArcGISProject("CURRENT")
            #print('Default geodatabase:',type(aprx.defaultGeodatabase))
            arcpy.Copy_management(aprx.defaultGeodatabase, output_gdb)  
            if map_name=='': mapa = aprx.listMaps()[0]
            else: mapa = aprx.listMaps(map_name)[0]
            #mapa.createGroupLayer(mapa.name)
            layers= mapa.listLayers()
            l=-1           
            for lyrx in layers:
                l+=1
                if lyrx.isFeatureLayer or lyrx.isRasterLayer:
                    new_list.append(lyrx.name)
                    #if l<10: lyrx.name= '_0'+str(l)+'_'+lyrx.name
                    #else: lyrx.name= '_'+str(l)+'_'+lyrx.name
                    #mapa.addLayerToGroup(layers[0],lyrx,'BOTTOM')
                    
                if lyrx.isGroupLayer:
                    print('group:',lyrx.name)
                    ch= arcpy.Describe(lyrx).children
                    print('childrens:\n')
                    for c in ch: print(c.baseName)
                    
                    print(type(lyrx))
                    arcpy.management.SaveToLayerFile(lyrx,output_folder+'\\'+lyrx.name+'.lyrx')
                    
            #aprx.save()
            
            
            #aprx.saveACopy(R"C:\Users\becari.alex.marcos\Documents\ArcGIS\Projects\testing\copy.aprx")
            #arcpy.management.SaveToLayerFile(layers[0],output_folder+'\\'+layers[0].name+'.lyrx')
                
            gdb_list= getGDB(aprx.defaultGeodatabase)
            
        arcpy.env.workspace= output_gdb
        if False:
            n=0
            for i in new_list:
                for file in gdb_list:
                    if i == file:
                        n+=1
                        if n<10:
                            arcpy.Rename_management(file, '_0'+str(n)+'_'+file)
                            print('New FC:', '_0'+str(n)+'_'+i)
                        else:
                            arcpy.Rename_management(file, '_'+str(n)+'_'+file)
                            print('New FC:', '_'+str(n)+'_'+i)
        
        gdb_vector= arcpy.ListFeatureClasses()
        gdb_rasters= arcpy.ListRasters()
        gdb_features= gdb_vector
        
        if False:
            output_gpkg= checkGPKG(output_folder, nom_gdb)
            if os.path.isfile(output_gpkg) == False:
                arcpy.management.CreateSQLiteDatabase(output_gpkg,'GEOPACKAGE_1.3' )
                print('GPKG creat')
            else: print('GPKG ',output_gpkg, ' already exists')

            for fc in gdb_features:
                print(fc)
                arcpy.management.CopyFeatures(output_gdb+'\\'+fc,output_gpkg+'\\'+fc)
                
    else: ('GDB ',output_gdb, ' already exists')


if __name__ == "__main__":
    
    env = R'C:\Users\becari.alex.marcos\Documents\ArcGIS\Projects\testing\testing.aprx'
    #env= R'C:\Users\becari.alex.marcos\OneDrive - Institut Cartogràfic i Geològic de Catalunya\done\gravimetria\gravimetria.aprx'
    carpeta = R'C:\Users\becari.alex.marcos\Documents\ArcGIS\Projects\testing'
    mapa = 'georef'
    nom_gdb= 'Mega'
    
    setName(env, nom_gdb, carpeta, mapa)

group: grav
childrens:

anomalia_bouguer_dades
anomalia_bouguer_500000
anomalia_bouguer_isolinies
anomalia_bouguer_250000
anomalia_regional_isolinies
anomalia_regional
anomalia_residual_isolinies
anomalia_residual
<class 'arcpy._mp.Layer'>
group: georef
childrens:

Plistocè_resto
Unitats2_ExportFeatures
T_02_anomalia_bouguer_500000
<class 'arcpy._mp.Layer'>
